## Bussiness case:

We’re opening a chocolate business and want to study our biggest competitor, from which we have access to their sales.

[Link for dataset](https://www.kaggle.com/datasets/arjunmehta1992/chocolate-sales-in-20222023/data)

We will look at different data to compare and prepare our go to market

We will shape the data in 3 tables: invoices, salesperson, product

We want to find out:

- Their prices
- Their discounts
- Seasonality of their sales
- Best salesperson to recruit for our company
- What country buys the most?

Viz:
- Leaderboard of sales
- Leaderboard countries

## About Dataset

This dataset captures order-level sales activity for a chocolate brand operating across multiple countries and sales channels between January 2022 and December 2023. Each row represents a single sales order, covering product details, discounting, marketing spend, and the resulting shipment volume.

The data reflects a mix of retail, online, and wholesale channels, with seasonal demand patterns (notably around Valentine's Day and the year-end holiday season) and variation across 25 salespeople, 7 countries, and 12 product lines.

As with most real-world sales exports, the data contains some inconsistencies — missing fields, duplicate records, mixed date formats, and a handful of outliers — that came through during the export/merge process and were left as-is.

Content

    Order_ID — Unique identifier for each order

    Product — Chocolate product name

    Country — Country of sale

    Channel — Sales channel (Retail / Online / Wholesale)

    Salesperson — Sales representative handling the order

    Order_Date — Date the order was placed

    Discount_Pct — Discount applied to the order (%)

    Price_per_Box — Price per box after discount (USD)

    Marketing_Spend — Marketing budget allocated around the order (USD)

    Boxes_Shipped — Number of boxes shipped for the order

    Amount — Total order value (Boxes_Shipped × Price_per_Box)


## Loading libraries

In [23]:
import pandas as pd
import numpy as np
import cleaning_chocolate as cc

%load_ext autoreload
%autoreload 2

## Loading dataframe

In [24]:
chocolate_df = pd.read_csv('Chocolate_Sales.csv')
chocolate_df

,Order_ID,Product,Country,Channel,Salesperson,Order_Date,Discount_Pct,Price_per_Box,Marketing_Spend,Boxes_Shipped,Amount
0,ORD-069833,Truffle Gift Box,Australia,Retail,Arjun Mehta,2022-12-11,3.5,13.72,202.03,71,912.31
1,ORD-090726,85% Dark Bar,Australia,Retail,Arjun Mehta,2023-03-14,9.4,3.30,55.18,84,245.91
2,ORD-042159,70% Dark Bar,Japan,Retail,Hannah Müller,2023-12-21,4.9,18.21,60.65,35,583.7
3,ORD-197166,Hazelnut Milk Bar,Germany,Retail,Arjun Mehta,2023-12-18,15.0,2.66,52.00,92,211.27
4,ORD-112162,Almond Crunch Bar,Australia,Retail,Yuki Sato,2023-08-18,4.4,2.75,187.44,214,549.69
...,...,...,...,...,...,...,...,...,...,...,...
199995,ORD-053460,70% Dark Bar,Australia,Retail,Beatriz Souza,2022-05-27,10.6,3.39,186.19,125,418.56
199996,ORD-010743,Truffle Gift Box,India,Wholesale,Arjun Mehta,2022-09-06,17.3,3.07,42.28,148,366.47
199997,ORD-049690,Salted Caramel Bar,Japan,Online,Arjun Mehta,2022-06-16,18.3,2.74,100.19,149,329.79
199998,ORD-189637,85% Dark Bar,Japan,Wholesale,Rohan Patel,2022-10-14,9.3,14.30,29.96,24,304.51


## Dataset issues:

### Missing values in the dataset:

order_date 437 rows -> drop as is 1% of the total rows

discount_pct 489 -> means no discount (add 0)

price_per_box 457 -> perhaps the values are somewhere else

marketing_spend 461 -> means no money spent (add 0)

#### Amount doesn't match price_per_box * boxes shipped. When adding the discount back it seems similar
- Fix: create a new amount column price_per_box * boxes_shipped
- Fix: rename old amount to amount_deprecated
- Fix: create a whole amount per box column (price per box+discount)

### Price per box is with price discounted. 
- Fix: Add back discount to new column to know what is the original price

### Fix box shipped when they're on minus





## Cleaning

In [25]:
#First, let's clean the column names of the chocolate DataFrame using the `clean_column_names` function from the 
# `cleaning_chocolate` module.

cleaned_chocolate_df = cc.clean_column_names(chocolate_df)
cleaned_chocolate_df

,order_id,product,country,channel,salesperson,order_date,discount_pct,price_per_box,marketing_spend,boxes_shipped,amount
0,ORD-069833,Truffle Gift Box,Australia,Retail,Arjun Mehta,2022-12-11,3.5,13.72,202.03,71,912.31
1,ORD-090726,85% Dark Bar,Australia,Retail,Arjun Mehta,2023-03-14,9.4,3.30,55.18,84,245.91
2,ORD-042159,70% Dark Bar,Japan,Retail,Hannah Müller,2023-12-21,4.9,18.21,60.65,35,583.7
3,ORD-197166,Hazelnut Milk Bar,Germany,Retail,Arjun Mehta,2023-12-18,15.0,2.66,52.00,92,211.27
4,ORD-112162,Almond Crunch Bar,Australia,Retail,Yuki Sato,2023-08-18,4.4,2.75,187.44,214,549.69
...,...,...,...,...,...,...,...,...,...,...,...
199995,ORD-053460,70% Dark Bar,Australia,Retail,Beatriz Souza,2022-05-27,10.6,3.39,186.19,125,418.56
199996,ORD-010743,Truffle Gift Box,India,Wholesale,Arjun Mehta,2022-09-06,17.3,3.07,42.28,148,366.47
199997,ORD-049690,Salted Caramel Bar,Japan,Online,Arjun Mehta,2022-06-16,18.3,2.74,100.19,149,329.79
199998,ORD-189637,85% Dark Bar,Japan,Wholesale,Rohan Patel,2022-10-14,9.3,14.30,29.96,24,304.51


In [26]:
cleaned_chocolate_df.sample(30)

,order_id,product,country,channel,salesperson,order_date,discount_pct,price_per_box,marketing_spend,boxes_shipped,amount
94947,ORD-084407,Milk Classic Bar,Brazil,Retail,Haruto Tanaka,2022-05-13,4.50,3.12,127.04,88,253.52
184694,ORD-026937,Mixed Assortment Box,Brazil,Retail,Arjun Mehta,2022-05-15,9.00,4.49,82.40,226,871.77
124960,ORD-158394,Almond Crunch Bar,Brazil,Wholesale,Arjun Mehta,2023-12-20,11.63,4.13,71.51,498,1934.92
48989,ORD-071483,70% Dark Bar,Australia,Wholesale,Gabriel Silva,2022-11-09,22.00,2.50,228.93,475,948.99
67569,ORD-150903,70% Dark Bar,Brazil,Retail,Rohan Patel,2022-08-22,23.90,2.96,175.21,253,530.63
28408,ORD-059712,70% Dark Bar,Australia,Retail,Priya Sharma,2023-03-24,3.10,3.27,77.56,93,290.62
72140,ORD-174946,85% Dark Bar,India,Retail,Emily Clarke,17/05/2022,8.10,3.03,126.88,56,157.19
25038,ORD-138718,70% Dark Bar,Brazil,Wholesale,Sophie Dubois,2022-08-21,18.00,13.20,54.92,50,509.19
94898,ORD-056784,85% Dark Bar,Brazil,Wholesale,Emily Clarke,2023-04-16,9.00,13.51,79.52,61,712.38
8757,ORD-119825,85% Dark Bar,Australia,Retail,Arjun Mehta,2023-11-30,23.50,2.97,67.55,164,375.1


In [27]:
negative_orders = (cleaned_chocolate_df["boxes_shipped"] < 0).sum()
negative_orders

np.int64(1956)

In [28]:
fixed_chocolate_df = cleaned_chocolate_df.copy()

In [29]:
fixed_chocolate_df = cc.create_price_columns(fixed_chocolate_df)

In [30]:
fixed_chocolate_df[[
    "discount_pct",
    "price_per_box_after_discount",
    "price_per_box_before_discount"
]].head(10).round(2)

,discount_pct,price_per_box_after_discount,price_per_box_before_discount
0,3.5,13.72,14.22
1,9.4,3.30,3.64
2,4.9,18.21,19.15
3,15.0,2.66,3.13
4,4.4,2.75,2.88
5,17.8,12.87,15.66
6,12.0,3.18,3.61
7,22.6,2.43,3.14
8,13.9,2.98,3.46
9,16.3,3.13,3.74


In [31]:
valid_prices = fixed_chocolate_df["price_per_box_after_discount"].notna()

(
    fixed_chocolate_df.loc[
        valid_prices, "price_per_box_before_discount"
    ]
    >=
    fixed_chocolate_df.loc[
        valid_prices, "price_per_box_after_discount"
    ]
).all()

np.True_

In [32]:
fixed_chocolate_df = cc.create_amount_columns(fixed_chocolate_df)

In [33]:
# Fill missing values in the 'discount_pct' and 'marketing_spend' columns with 0.
cleaned_chocolate_df = cc.fill_missing_with_zero(cleaned_chocolate_df, 'discount_pct')
cleaned_chocolate_df = cc.fill_missing_with_zero(cleaned_chocolate_df, 'marketing_spend')
cleaned_chocolate_df

,discount_pct,price_per_box_before_discount,price_per_box_after_discount,boxes_shipped,amount_before_discount,total_discount,amount_after_discount,amount_deprecated
0,3.5,14.217617,13.72,71,1009.45,35.33,974.12,912.31
1,9.4,3.642384,3.30,84,305.96,28.76,277.20,245.91
2,4.9,19.148265,18.21,35,670.19,32.84,637.35,583.7
3,15.0,3.129412,2.66,92,287.91,43.19,244.72,211.27
4,4.4,2.876569,2.75,214,615.59,27.09,588.50,549.69
5,17.8,15.656934,12.87,47,735.88,130.99,604.89,527.01
6,12.0,3.613636,3.18,159,574.57,68.95,505.62,470.63
7,22.6,3.139535,2.43,326,1023.49,231.31,792.18,629.51
8,13.9,3.461092,2.98,70,242.28,33.68,208.60,181.16
9,16.3,3.739546,3.13,213,796.52,129.83,666.69,588.18
